In [ ]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
import torch

In [ ]:
data = pd.read_csv('data/SU.csv', sep=';')
print(data.columns)

In [ ]:
for name in list(data['name.2'].unique()):
    print(name)

In [ ]:
# keep most recent row per course based on start_date
selected_columns = data[['identifier','start_date','name.2','name','plan_description','objectives']]
selected_columns['start_date'] = pd.to_datetime(selected_columns['start_date'], errors='coerce')
selected_columns = (selected_columns.sort_values(['identifier', 'start_date'], ascending=[True, False])
                                  .drop_duplicates(subset='identifier', keep='first')
                                  .reset_index(drop=True))
selected_columns.to_csv('data/SU_only_most_recent.csv', sep=";", index=False)

In [ ]:
file = pd.read_csv('data/SU_only_most_recent.csv', sep=';')
file.head()

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
sentences = ["Det här är en exempelmening", "Det här är en annan exempelmening"]

model = SentenceTransformer('KBLab/sentence-bert-swedish-cased')
embeddings = model.encode(sentences)
print(embeddings)

similarity = embeddings[0] @ embeddings[1] / (np.linalg.norm(embeddings[0]) * np.linalg.norm(embeddings[1]))
print(f"Cosine similarity between the two sentences: {similarity}")

In [ ]:
plans = file['plan_description'].tolist()
plans_embed = dict(zip(plans, model.encode(plans)))

In [ ]:
query = "redogöra för medierättens komplicerade regelstruktur och systematik och dess rättskällor"
query_embed = model.encode(query)
similarities = {plan: query_embed @ embed / (np.linalg.norm(query_embed) * np.linalg.norm(embed)) for plan, embed in plans_embed.items()}
sorted_plans = sorted(similarities.items(), key=lambda x: x[1], reverse=True)
print("Top 5 most similar plans:")
for plan, sim in sorted_plans[:5]:
    print(f"Plan: {plan}, Similarity: {sim}")

In [ ]:
sentences = [
    "I denna kurs kommer studenten lära sig att programmera i Python",
    "utveckla en förståelse för programmering i Python",
    "förståelse för aktuella metoder i tillämpad matematik"
]
embeddings = model.encode(sentences)
similarity = torch.cosine_similarity(torch.tensor(embeddings[0]), torch.tensor(embeddings[1]), dim=0)
print(f"Cosine similarity between the first two sentences: {similarity.item()}")

In [ ]:
class Outcome:
    def __init__(self, outcome, code):
        self.outcome = outcome
        self.code = code
        self.embedding = model.encode(outcome)
    
    def __str__(self):
        return self.outcome

In [ ]:
class Plan:
    def __init__(self, description, code):
        self.description = description
        self.code = code
        self.embedding = model.encode(description)

    def __str__(self):
        return self.description

In [ ]:
model = SentenceTransformer('KBLab/sentence-bert-swedish-cased')

In [ ]:
import json 

with open('data/SU.heuristics.json', 'r') as f:
    corpus = json.load(f)

outcomes = []
plans = []

for item in corpus['Course-list']:
    plans.append(Plan(item['CourseContent'], item['CourseCode']))
    for outcome in item['ILO-list-sv']:
        outcomes.append(Outcome(outcome, item['CourseCode']))
    break

In [1]:
import json 

with open('data/SU.heuristics.json', 'r') as f:
    corpus = json.load(f)

codes = set()
i = 0
for item in corpus['Course-list']:
    codes.add(item['CourseCode'])
    i += 1

print(i)
print(len(codes))


4880
4880


In [ ]:
print(outcomes[0].outcome)
print(outcomes[0].embedding)

In [ ]:
from transformers import AutoModel,AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('KBLab/bert-base-swedish-cased')
model = AutoModel.from_pretrained('KBLab/bert-base-swedish-cased')

In [ ]:
# USING BERT'S CLS TOKEN FOR SIMILARITY CALCULATION
import torch
import torch.nn.functional as F

sentence1 = "I denna kurs kommer studenten lära sig att programmera i Python"
sentence2 = "utveckla en förståelse för programmering i Python"
control_sentence = "förståelse för aktuella metoder i tillämpad matematik"
inputs1 = tokenizer(sentence1, return_tensors='pt')
outputs1 = model(**inputs1)
inputs2 = tokenizer(sentence2, return_tensors='pt')
outputs2 = model(**inputs2)
inputs3 = tokenizer(control_sentence, return_tensors='pt')
outputs3 = model(**inputs3)

cosine_similarity = (outputs1.last_hidden_state[0][0] @ outputs2.last_hidden_state[0][0]) / (torch.norm(outputs1.last_hidden_state[0][0]) * torch.norm(outputs2.last_hidden_state[0][0]))
print(f"Cosine similarity between sentences 1 - 2: {cosine_similarity.item()}")

cosine_similarity_control = (outputs1.last_hidden_state[0][0] @ outputs3.last_hidden_state[0][0]) / (torch.norm(outputs1.last_hidden_state[0][0]) * torch.norm(outputs3.last_hidden_state[0][0]))
print(f"Cosine similarity between sentence 1 - control: {cosine_similarity_control.item()}")

In [ ]:
import torch
import torch.nn.functional as F

def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output.last_hidden_state  # (batch, seq_len, hidden)
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return (token_embeddings * input_mask_expanded).sum(dim=1) / input_mask_expanded.sum(dim=1)


In [ ]:
model.eval()

sentences = [
    "I denna kurs kommer studenten lära sig att programmera i Python",
    "utveckla en förståelse för programmering i Python",
    "förståelse för aktuella metoder i tillämpad matematik"
]

with torch.no_grad():
    encoded = tokenizer(
        sentences,
        padding=True,
        truncation=True,
        return_tensors="pt"
    )
    outputs = model(**encoded)


In [ ]:
sentence_embeddings = mean_pooling(outputs, encoded['attention_mask'])
sentence_embeddings = F.normalize(sentence_embeddings, p=2, dim=1)


In [ ]:
sim_1_2 = torch.cosine_similarity(
    sentence_embeddings[0],
    sentence_embeddings[1],
    dim=0
)

sim_1_control = torch.cosine_similarity(
    sentence_embeddings[0],
    sentence_embeddings[2],
    dim=0
)

print(f"Cosine similarity (1–2): {sim_1_2.item():.4f}")
print(f"Cosine similarity (1–control): {sim_1_control.item():.4f}")


In [8]:
import torch

t1 = torch.tensor([1.0, 2.0, 3.0])
t2 = torch.tensor([1.0, 2.0, 3.0])
similarity = torch.cosine_similarity(t1, t2, dim=0)
print(f"Cosine similarity: {similarity.item()}")
type(similarity.item())

Cosine similarity: 0.9999998807907104


float